#  Evaluation



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/prajwl7676/mini-clip-DLAI.git

%cd /content/mini-clip-DLAI
!pip install -q -r requirements.txt

import sys, os
sys.path.insert(0, '/content/mini-clip-DLAI')

In [ ]:
%cd /content/mini-clip-DLAI

In [ ]:
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import DistilBertTokenizer

from src.model_architecture  import CLIPModel
from src.dataset  import FlickrDataset, get_transform, build_loaders
from src.train    import load_checkpoint
from src.evaluate import encode_dataset, recall_at_k, print_results, retrieval_examples

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
BEST_CKPT = '/content/drive/MyDrive/mini-clip-dlai/checkpoints/best.pt'
EMBEDDING_DIM = 256
BATCH_SIZE    = 64

---
##  Load data (test split only)

In [ ]:
full_ds   = load_dataset('nlphuji/flickr30k', split='test')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

test_hf   = full_ds.select(range(9_000, 10_000))   # same 1000 as training split
test_ds   = FlickrDataset(test_hf, tokenizer, get_transform(train=False))
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
print(f'Test set: {len(test_ds)} samples  →  {len(test_loader)} batches')

---
##  Load best checkpoint

In [ ]:
model     = CLIPModel(embedding_dim=EMBEDDING_DIM).to(device)

projection_params = (
    list(model.image_encoder.projection.parameters()) +
    list(model.text_encoder.projection.parameters())  +
    [model.log_temperature]
)
encoder_params = [
    p for p in model.parameters()
    if p.requires_grad and not any(p is pp for pp in projection_params)
]

optimizer = torch.optim.AdamW([
    {'params': encoder_params,    'lr': 1e-4},
    {'params': projection_params, 'lr': 1e-3},
])

ckpt = load_checkpoint(model, optimizer, BEST_CKPT, device)

print(f'Loaded checkpoint from epoch {ckpt["epoch"] + 1}')
print(f'Val loss at best epoch : {ckpt["val_loss"]:.4f}')
print(f'Temperature τ          : {model.temperature.item():.4f}')

---
##  Encode the full test set


In [ ]:
print('Encoding test set...')
embeddings = encode_dataset(model, test_loader, device)

img_embs = embeddings['image_embeddings']   # (1000, 256)
txt_embs = embeddings['text_embeddings']    # (1000, 256)

print(f'Image embeddings : {img_embs.shape}')
print(f'Text  embeddings : {txt_embs.shape}')

# Verify L2-normalisation
print(f'Image norms (should be ~1.0): mean={img_embs.norm(dim=1).mean():.4f}')
print(f'Text  norms (should be ~1.0): mean={txt_embs.norm(dim=1).mean():.4f}')

---
##  Compute Recall@K (fine-tuned model)


In [ ]:
results_finetuned = recall_at_k(img_embs, txt_embs, ks=[1, 5, 10])
print_results(results_finetuned, title='Fine-tuned mini-CLIP (best.pt)')

---
##  Baseline: frozen pretrained encoders (no fine-tuning)


In [ ]:
baseline_model = CLIPModel(embedding_dim=EMBEDDING_DIM).to(device)
baseline_model.eval()

print('Encoding test set with baseline (no fine-tuning)...')
base_embs = encode_dataset(baseline_model, test_loader, device)

results_baseline = recall_at_k(
    base_embs['image_embeddings'],
    base_embs['text_embeddings'],
    ks=[1, 5, 10]
)
print_results(results_baseline, title='Baseline (frozen pretrained, no fine-tuning)')

---
##  Results comparison table

In [ ]:
print('\n=== Results Summary ===')
print(f'{"Metric":<14} {"Baseline":>10} {"Fine-tuned":>12} {"Δ gain":>8}')
print('─' * 48)
for direction, label in [("t2i", "Text→Image"), ("i2t", "Image→Text")]:
    print(f'  {label}')
    for k in [1, 5, 10]:
        key  = f'{direction}_R@{k}'
        base = results_baseline[key]
        fine = results_finetuned[key]
        gain = fine - base
        sign = '+' if gain >= 0 else ''
        print(f'    R@{k:<4}  {base:>9.2f}%  {fine:>11.2f}%  {sign}{gain:>6.2f}%')
print('─' * 48)

---
##  Plot results bar chart

In [ ]:
import numpy as np

metrics = ['R@1', 'R@5', 'R@10']
ks      = [1, 5, 10]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, (direction, title) in zip(axes, [('t2i', 'Text → Image'), ('i2t', 'Image → Text')]):
    base_scores = [results_baseline[f'{direction}_R@{k}'] for k in ks]
    fine_scores = [results_finetuned[f'{direction}_R@{k}'] for k in ks]

    x     = np.arange(len(ks))
    width = 0.35

    bars1 = ax.bar(x - width/2, base_scores, width,
                   label='Baseline', color='#B0BEC5', edgecolor='white')
    bars2 = ax.bar(x + width/2, fine_scores, width,
                   label='Fine-tuned', color='#1D9E75', edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylabel('Recall (%)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_ylim(0, max(fine_scores) * 1.25)

    for bar in bars1 + bars2:
        ax.annotate(
            f'{bar.get_height():.1f}',
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3), textcoords='offset points',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle('Retrieval Performance: Baseline vs Fine-tuned Mini-CLIP', fontsize=12)
plt.tight_layout()
plt.savefig('assets/recall_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/recall_results.png')

---
##  Qualitative retrieval examples

In [ ]:
retrieval_examples(
    model        = model,
    test_hf_dataset = test_hf,
    tokenizer    = tokenizer,
    get_transform_fn = get_transform,
    device       = device,
    query_indices = [0, 1, 2, 3],
    top_k        = 5,
    save_path    = 'assets/retrieval_examples.png',
)